## GPT2

In [7]:
layer_name_map = {
    "wte.weight": "wte",
    "wpe.weight": "wpe",
}

block_map = {
    "h.{i}.ln_1.weight": "h.{i}.ln1.weight",
    "h.{i}.ln_1.bias": "h.{i}.ln1.bias",
    "h.{i}.attn.bias": "h.{i}.attn.causal.mask",
    "h.{i}.attn.c_attn.weight": "h.{i}.qkv.weight",
    "h.{i}.attn.c_attn.bias": "h.{i}.qkv.bias",
    "h.{i}.attn.c_proj.weight": "h.{i}.attn.proj.weight",
    "h.{i}.attn.c_proj.bias": "h.{i}.attn.proj.bias",
    "h.{i}.ln_2.weight": "h.{i}.ln2.weight",
    "h.{i}.ln_2.bias": "h.{i}.ln2.bias",
    "h.{i}.mlp.c_fc.weight": "h.{i}.mlp.up.weight",
    "h.{i}.mlp.c_fc.bias": "h.{i}.mlp.up.bias",
    "h.{i}.mlp.c_proj.weight": "h.{i}.mlp.down.weight",
    "h.{i}.mlp.c_proj.bias": "h.{i}.mlp.down.bias",
}

for i in range(12):
    for src, dst in block_map.items():
        layer_name_map[src.format(i=i)] = dst.format(i=i)

layer_name_map["ln_f.weight"] = "ln.weight"
layer_name_map["ln_f.bias"] = "ln.bias"

layer_name_map

{'wte.weight': 'wte',
 'wpe.weight': 'wpe',
 'h.0.ln_1.weight': 'h.0.ln1.weight',
 'h.0.ln_1.bias': 'h.0.ln1.bias',
 'h.0.attn.bias': 'h.0.attn.causal.mask',
 'h.0.attn.c_attn.weight': 'h.0.qkv.weight',
 'h.0.attn.c_attn.bias': 'h.0.qkv.bias',
 'h.0.attn.c_proj.weight': 'h.0.attn.proj.weight',
 'h.0.attn.c_proj.bias': 'h.0.attn.proj.bias',
 'h.0.ln_2.weight': 'h.0.ln2.weight',
 'h.0.ln_2.bias': 'h.0.ln2.bias',
 'h.0.mlp.c_fc.weight': 'h.0.mlp.up.weight',
 'h.0.mlp.c_fc.bias': 'h.0.mlp.up.bias',
 'h.0.mlp.c_proj.weight': 'h.0.mlp.down.weight',
 'h.0.mlp.c_proj.bias': 'h.0.mlp.down.bias',
 'h.1.ln_1.weight': 'h.1.ln1.weight',
 'h.1.ln_1.bias': 'h.1.ln1.bias',
 'h.1.attn.bias': 'h.1.attn.causal.mask',
 'h.1.attn.c_attn.weight': 'h.1.qkv.weight',
 'h.1.attn.c_attn.bias': 'h.1.qkv.bias',
 'h.1.attn.c_proj.weight': 'h.1.attn.proj.weight',
 'h.1.attn.c_proj.bias': 'h.1.attn.proj.bias',
 'h.1.ln_2.weight': 'h.1.ln2.weight',
 'h.1.ln_2.bias': 'h.1.ln2.bias',
 'h.1.mlp.c_fc.weight': 'h.1.mlp.up.

In [8]:
import json
import struct

path = "/Users/uonliaquat/Downloads/gpt2.safetensors"

with open(path, "rb") as f:
    header_size = struct.unpack("<Q", f.read(8))[0]
    header = json.loads(f.read(header_size))

data_start = 8 + header_size

with open("./gpt2.zg", "w") as gpt2_zg:
    for name, info in header.items():
        if name == "__metadata__":
            continue

        name = layer_name_map[name]

        start, end = info["data_offsets"]

        start += data_start
        end += data_start

        nbytes = end - start

        gpt2_zg.write(
            f"{name:<40}{start},{end}\n"
        )

        print(
            f"{name:<50} start={start:<12} end={end:<12} nbytes={nbytes}"
        )

h.3.ln2.bias                                       start=202434515    end=202437587    nbytes=3072
h.10.ln1.weight                                    start=223168467    end=223171539    nbytes=3072
h.2.qkv.weight                                     start=541027283    end=548105171    nbytes=7077888
h.4.attn.proj.weight                               start=18968531     end=21327827     nbytes=2359296
h.7.attn.causal.mask                               start=263353299    end=267547603    nbytes=4194304
h.4.qkv.bias                                       start=150989779    end=150998995    nbytes=9216
h.2.attn.causal.mask                               start=516095955    end=520290259    nbytes=4194304
h.5.mlp.down.bias                                  start=21327827     end=21330899     nbytes=3072
h.1.attn.proj.weight                               start=513730515    end=516089811    nbytes=2359296
h.4.ln1.weight                                     start=206638035    end=206641107    nbytes=

## Compare Zg and HF weights

In [9]:
from safetensors.torch import load_file

hf_path = "/Users/uonliaquat/Downloads/gpt2.safetensors"
zg_path = "/Users/uonliaquat/workspace/zerograd/my_model.safetensors"

tensors_hf = load_file(hf_path)
tensors_zg = load_file(zg_path)

max_diff_overall = -10
for key_hf, key_zg in layer_name_map.items():
    # key_hf = "wte.weight"
    # key_zg = layer_name_map[key_hf]

    # print(f"key_hf: {key_hf}, key_zg: {key_zg}\n\n")

    if key_hf not in tensors_hf:
        print(f"{key_hf} not found in tensors_hf\n")
        continue
    if key_zg not in tensors_zg:
        print(f"{key_zg} not found in tensors_zg\n")
        continue

    t_hf = tensors_hf[key_hf]
    t_zg = tensors_zg[key_zg]

    print(f"Name: t_hf:         {key_hf}    | t_zg: {key_zg}")
    print(f"hf.shape:           {list(t_hf.shape)}  | zg.shape: {list(t_zg.shape)}")
    print(f"Are shapes equal:   {list(t_hf.shape) == list(t_zg.shape)}")
    diff = abs((t_hf.flatten() - t_zg.flatten())).max().item()
    print(f"Diff of data:       {diff}\n")
    if diff > max_diff_overall:
        max_diff_overall = diff

print(f"\n\nmax_diff_overall: {max_diff_overall}")

Name: t_hf:         wte.weight    | t_zg: wte
hf.shape:           [50257, 768]  | zg.shape: [50257, 768]
Are shapes equal:   True
Diff of data:       0.0

Name: t_hf:         wpe.weight    | t_zg: wpe
hf.shape:           [1024, 768]  | zg.shape: [1024, 768]
Are shapes equal:   True
Diff of data:       0.0

Name: t_hf:         h.0.ln_1.weight    | t_zg: h.0.ln1.weight
hf.shape:           [768]  | zg.shape: [768]
Are shapes equal:   True
Diff of data:       0.0

Name: t_hf:         h.0.ln_1.bias    | t_zg: h.0.ln1.bias
hf.shape:           [768]  | zg.shape: [768]
Are shapes equal:   True
Diff of data:       0.0

h.0.attn.causal.mask not found in tensors_zg

Name: t_hf:         h.0.attn.c_attn.weight    | t_zg: h.0.qkv.weight
hf.shape:           [768, 2304]  | zg.shape: [768, 2304]
Are shapes equal:   True
Diff of data:       0.0

Name: t_hf:         h.0.attn.c_attn.bias    | t_zg: h.0.qkv.bias
hf.shape:           [2304]  | zg.shape: [2304]
Are shapes equal:   True
Diff of data:       0.0

In [23]:
from safetensors.torch import load_file

hf_path = "/Users/uonliaquat/Downloads/gpt2.safetensors"
zg_path = "/Users/uonliaquat/workspace/zerograd/my_model.safetensors"

tensors_hf = load_file(hf_path)
tensors_zg = load_file(zg_path)

key_hf = "h.0.attn.c_proj.weight"
key_zg = layer_name_map[key_hf]

# print(f"key_hf: {key_hf}, key_zg: {key_zg}\n\n")

if key_hf not in tensors_hf:
    print("key_hf not found in tensors_hf\n")
if key_zg not in tensors_zg:
    print("key_zg not found in tensors_zg\n")

t_hf = tensors_hf[key_hf]
t_zg = tensors_zg[key_zg]

print(f"Name: t_hf:         {key_hf}    | t_zg: {key_zg}")
print(f"hf.shape:           {list(t_hf.shape)}  | zg.shape: {list(t_zg.shape)}")
print(f"Are shapes equal:   {list(t_hf.shape) == list(t_zg.shape)}")
print(f"Diff of data:       {(t_hf.flatten() - t_zg.flatten()).max().item()}")
print(f"Data (hf):          {t_hf.flatten()[:5]}")
print(f"Data (zg):          {t_zg.flatten()[:5]}")


Name: t_hf:         h.0.attn.c_proj.weight    | t_zg: h.0.attn.proj.weight
hf.shape:           [768, 768]  | zg.shape: [768, 768]
Are shapes equal:   True
Diff of data:       0.0
Data (hf):          tensor([ 0.3127, -0.1874,  0.0980, -0.0303, -0.0216])
Data (zg):          tensor([ 0.3127, -0.1874,  0.0980, -0.0303, -0.0216])


In [3]:
import math
import torch
from transformers import GPT2Model
from safetensors.torch import load_file

path = "/Users/uonliaquat/Downloads/gpt2.safetensors"

model = GPT2Model.from_pretrained("gpt2")
model.load_state_dict(load_file(path), strict=False)
model.eval()


def dump(name, x):
    x = x.detach().cpu()
    print(f"\n{name}  shape={tuple(x.shape)}")
    x = x.flatten()
    print(", ".join(f"{v:.3f}" for v in x[:10]))


def dump_op(name, output, weight=None, bias=None):
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    if weight is not None:
        dump(f"{name}.weight", weight)

    if bias is not None:
        dump(f"{name}.bias", bias)

    dump(f"{name}.output", output)


# Use >1 token so the causal mask and softmax are actually exercised.
# Make sure your C code feeds these exact same ids in the same order.
input_ids = torch.zeros((1, 1024), dtype=torch.long)
seq_len = input_ids.shape[1]

with torch.no_grad():

    # ------------------------------------------------------------------
    # embeddings
    # ------------------------------------------------------------------

    dump("wte.weight", model.wte.weight)

    tok = model.wte(input_ids)
    dump("wte.output", tok)

    dump("wpe.weight", model.wpe.weight)

    pos_ids = torch.arange(seq_len).unsqueeze(0)
    pos = model.wpe(pos_ids)
    dump("wpe.output", pos)

    x = tok + pos
    dump("input.embed", x)

    block = model.h[0]

    # ------------------------------------------------------------------
    # LN1
    # ------------------------------------------------------------------

    ln1 = block.ln_1(x)

    dump_op(
        "h.0.ln1",
        ln1,
        block.ln_1.weight,
        block.ln_1.bias
    )

    # ------------------------------------------------------------------
    # QKV
    #   c_attn is a Conv1D: weight stored as (in, out) = (768, 2304).
    #   Dumped WITHOUT .T to match raw row-major safetensors bytes.
    # ------------------------------------------------------------------

    qkv = block.attn.c_attn(ln1)

    dump_op(
        "h.0.qkv",
        qkv,
        block.attn.c_attn.weight,
        block.attn.c_attn.bias
    )

    hidden = qkv.shape[-1] // 3

    q = qkv[:, :, :hidden]
    k = qkv[:, :, hidden:2 * hidden]
    v = qkv[:, :, 2 * hidden:]

    dump("h.0.q", q)
    dump("h.0.k", k)
    dump("h.0.v", v)

    # ------------------------------------------------------------------
    # reshape heads  (generalized to seq_len)
    # ------------------------------------------------------------------

    n_heads = block.attn.num_heads
    head_dim = hidden // n_heads

    qh = q.view(1, seq_len, n_heads, head_dim).transpose(1, 2)
    kh = k.view(1, seq_len, n_heads, head_dim).transpose(1, 2)
    vh = v.view(1, seq_len, n_heads, head_dim).transpose(1, 2)

    dump("h.0.q.heads", qh)
    dump("h.0.k.heads", kh)
    dump("h.0.v.heads", vh)

    # ------------------------------------------------------------------
    # QK^T
    # ------------------------------------------------------------------

    scores = torch.matmul(qh, kh.transpose(-2, -1))
    dump("h.0.attn.scores", scores)

    # ------------------------------------------------------------------
    # scale
    # ------------------------------------------------------------------

    scores = scores / math.sqrt(head_dim)
    dump("h.0.attn.scaled_scores", scores)

    # ------------------------------------------------------------------
    # causal mask
    # ------------------------------------------------------------------

    mask = torch.tril(torch.ones(seq_len, seq_len))
    scores = scores.masked_fill(mask == 0, float("-inf"))

    dump("h.0.attn.masked_scores", scores)

    # ------------------------------------------------------------------
    # softmax
    # ------------------------------------------------------------------

    probs = torch.softmax(scores, dim=-1)
    dump("h.0.attn.softmax", probs)

    # ------------------------------------------------------------------
    # attention @ value
    # ------------------------------------------------------------------

    attn = torch.matmul(probs, vh)
    dump("h.0.attn.weighted_values", attn)

    # ------------------------------------------------------------------
    # merge heads
    # ------------------------------------------------------------------

    attn = (
        attn.transpose(1, 2)
        .contiguous()
        .view(1, seq_len, hidden)
    )

    dump("h.0.attn.merge_heads", attn)

    # ------------------------------------------------------------------
    # output projection  (Conv1D, (in, out) = (768, 768), no .T)
    # ------------------------------------------------------------------

    attn_proj = block.attn.c_proj(attn)

    dump_op(
        "h.0.attn.proj",
        attn_proj,
        block.attn.c_proj.weight,
        block.attn.c_proj.bias
    )

    # ------------------------------------------------------------------
    # residual
    # ------------------------------------------------------------------

    resid1 = x + attn_proj
    dump("h.0.residual1", resid1)

    # ------------------------------------------------------------------
    # LN2
    # ------------------------------------------------------------------

    ln2 = block.ln_2(resid1)

    dump_op(
        "h.0.ln2",
        ln2,
        block.ln_2.weight,
        block.ln_2.bias
    )

    # ------------------------------------------------------------------
    # MLP up  (Conv1D, (in, out) = (768, 3072), no .T)
    # ------------------------------------------------------------------

    mlp_up = block.mlp.c_fc(ln2)

    dump_op(
        "h.0.mlp.up",
        mlp_up,
        block.mlp.c_fc.weight,
        block.mlp.c_fc.bias
    )

    # ------------------------------------------------------------------
    # GELU  (gpt2 uses the tanh approximation = gelu_new)
    #   C must use: 0.5*x*(1+tanh(sqrt(2/pi)*(x+0.044715*x^3)))
    # ------------------------------------------------------------------

    gelu = block.mlp.act(mlp_up)
    dump("h.0.mlp.gelu", gelu)

    # ------------------------------------------------------------------
    # MLP down  (Conv1D, (in, out) = (3072, 768), no .T)  <-- fixed
    # ------------------------------------------------------------------

    mlp_down = block.mlp.c_proj(gelu)

    dump_op(
        "h.0.mlp.down",
        mlp_down,
        block.mlp.c_proj.weight,   # was .T before — removed for consistency
        block.mlp.c_proj.bias
    )

    # ------------------------------------------------------------------
    # residual
    # ------------------------------------------------------------------

    resid2 = resid1 + mlp_down
    dump("h.0.residual2", resid2)

    # ------------------------------------------------------------------
    # final LN
    # ------------------------------------------------------------------

    final_ln = model.ln_f(resid2)

    dump_op(
        "ln_f",
        final_ln,
        model.ln_f.weight,
        model.ln_f.bias
    )


wte.weight  shape=(50257, 768)
-0.110, -0.039, 0.033, 0.134, -0.048, -0.079, -0.240, -0.089, 0.025, -0.107

wte.output  shape=(1, 1024, 768)
-0.110, -0.039, 0.033, 0.134, -0.048, -0.079, -0.240, -0.089, 0.025, -0.107

wpe.weight  shape=(1024, 768)
-0.019, -0.197, 0.004, 0.011, 0.064, -0.105, 0.037, -0.168, -0.049, -0.056

wpe.output  shape=(1, 1024, 768)
-0.019, -0.197, 0.004, 0.011, 0.064, -0.105, 0.037, -0.168, -0.049, -0.056

input.embed  shape=(1, 1024, 768)
-0.129, -0.237, 0.037, 0.145, 0.015, -0.184, -0.203, -0.258, -0.024, -0.164

==================== h.0.ln1 ====================

h.0.ln1.weight  shape=(768,)
0.223, 0.182, 0.153, 0.192, 0.204, 0.195, 0.147, 0.187, 0.214, 0.196

h.0.ln1.bias  shape=(768,)
-0.004, 0.027, -0.064, -0.005, -0.016, -0.011, 0.202, 0.036, -0.002, -0.008

h.0.ln1.output  shape=(1, 1024, 768)
-0.079, -0.087, -0.047, 0.073, -0.004, -0.106, 0.123, -0.092, -0.013, -0.093

==================== h.0.qkv ====================

h.0.qkv.weight  shape=(768, 2304)
-

In [5]:
import numpy as np

def layernorm(x, weight, bias, eps=1e-5):
    """
    x      : input vector [hidden_dim]
    weight : gamma [hidden_dim]
    bias   : beta  [hidden_dim]
    """

    mean = np.mean(x)
    var = np.mean((x - mean) ** 2)

    x_norm = (x - mean) / np.sqrt(var + eps)

    return x_norm * weight + bias


# Example
x = np.array([-0.129, -0.237, 0.037, 0.145, 0.015, -0.184, -0.203, -0.258, -0.024, -0.164], dtype=np.float32)

weight = np.array([0.223, 0.182, 0.153, 0.192, 0.204, 0.195, 0.147, 0.187, 0.214, 0.196], dtype=np.float32)
bias   = np.array([-0.004, 0.027, -0.064, -0.005, -0.016, -0.011, 0.202, 0.036, -0.002, -0.008], dtype=np.float32)

y = layernorm(x, weight, bias)

print(y)

[-0.05411544 -0.16728164  0.09980222  0.36236346  0.16738208 -0.13851252
  0.08408076 -0.19426231  0.12524565 -0.10557781]


In [10]:
from safetensors import safe_open

path = "/Users/uonliaquat/workspace/zerograd/my_model.safetensors"


with safe_open(path, framework="pt", device="cpu") as f:
    print(f"Number of tensors: {len(f.keys())}\n")

    for name in f.keys():
        tensor = f.get_tensor(name)
        print(f"{name:<40} shape={list(tensor.shape)}")

Number of tensors: 275

h.0.attn.out                             shape=[10, 768]
h.0.attn.proj                            shape=[10, 768]
h.0.attn.proj.bias                       shape=[768]
h.0.attn.proj.weight                     shape=[768, 768]
h.0.ln1.bias                             shape=[768]
h.0.ln1.out                              shape=[10, 768]
h.0.ln1.weight                           shape=[768]
h.0.ln2.bias                             shape=[768]
h.0.ln2.out                              shape=[10, 768]
h.0.ln2.weight                           shape=[768]
h.0.mlp.down.bias                        shape=[768]
h.0.mlp.down.proj                        shape=[10, 768]
h.0.mlp.down.weight                      shape=[3072, 768]
h.0.mlp.up.bias                          shape=[3072]
h.0.mlp.up.proj                          shape=[10, 3072]
h.0.mlp.up.proj.gelu.out                 shape=[10, 3072]
h.0.mlp.up.weight                        shape=[768, 3072]
h.0.out                    

In [ ]:
import math
import torch
from transformers import GPT2Model
from safetensors.torch import load_file
from safetensors import safe_open

# ------------------------------------------------------------------
# Mapping HF -> Zerograd
# ------------------------------------------------------------------

layer_name_map = {
    "wte.weight": "wte",
    "wpe.weight": "wpe",
}

block_map = {
    "h.{i}.ln_1.weight": "h.{i}.ln1.weight",
    "h.{i}.ln_1.bias": "h.{i}.ln1.bias",
    "h.{i}.attn.bias": "h.{i}.attn.causal.mask",
    "h.{i}.attn.c_attn.weight": "h.{i}.qkv.weight",
    "h.{i}.attn.c_attn.bias": "h.{i}.qkv.bias",
    "h.{i}.attn.c_proj.weight": "h.{i}.attn.proj.weight",
    "h.{i}.attn.c_proj.bias": "h.{i}.attn.proj.bias",
    "h.{i}.ln_2.weight": "h.{i}.ln2.weight",
    "h.{i}.ln_2.bias": "h.{i}.ln2.bias",
    "h.{i}.mlp.c_fc.weight": "h.{i}.mlp.up.weight",
    "h.{i}.mlp.c_fc.bias": "h.{i}.mlp.up.bias",
    "h.{i}.mlp.c_proj.weight": "h.{i}.mlp.down.weight",
    "h.{i}.mlp.c_proj.bias": "h.{i}.mlp.down.bias",
}

for i in range(12):
    for src, dst in block_map.items():
        layer_name_map[src.format(i=i)] = dst.format(i=i)

layer_name_map["ln_f.weight"] = "ln.weight"
layer_name_map["ln_f.bias"] = "ln.bias"

# ------------------------------------------------------------------
# Load HF GPT2
# ------------------------------------------------------------------

hf_path = "/Users/uonliaquat/Downloads/gpt2.safetensors"

model = GPT2Model.from_pretrained("gpt2")
model.load_state_dict(load_file(hf_path), strict=False)
model.eval()

# ------------------------------------------------------------------
# Load Zerograd tensors
# ------------------------------------------------------------------

zg_path = "/Users/uonliaquat/workspace/zerograd/my_model.safetensors"

zg = {}

with safe_open(zg_path, framework="pt", device="cpu") as f:
    for k in f.keys():
        zg[k] = f.get_tensor(k)

# ------------------------------------------------------------------
# Compare helper
# ------------------------------------------------------------------

def compare_weight(hf_name, hf_tensor):
    zg_name = layer_name_map[hf_name]

    if zg_name not in zg:
        print(f"\nMISSING: {zg_name}")
        return

    zg_tensor = zg[zg_name]

    hf_flat = hf_tensor.detach().cpu().flatten().float()
    zg_flat = zg_tensor.detach().cpu().flatten().float()

    print("\n" + "=" * 80)
    print(f"HF : {hf_name}")
    print(f"ZG : {zg_name}")
    print(f"HF shape : {tuple(hf_tensor.shape)}")
    print(f"ZG shape : {tuple(zg_tensor.shape)}")

    print("\nHF first 10:")
    print(", ".join(f"{x:.6f}" for x in hf_flat[:10]))

    print("\nZG first 10:")
    print(", ".join(f"{x:.6f}" for x in zg_flat[:10]))

    if hf_flat.numel() == zg_flat.numel():
        diff = (hf_flat - zg_flat).abs().max().item()
        print(f"\nMAX DIFF: {diff:e}")
    else:
        print("\nSIZE MISMATCH")


# ------------------------------------------------------------------
# Embeddings
# ------------------------------------------------------------------

compare_weight(
    "wte.weight",
    model.wte.weight
)

compare_weight(
    "wpe.weight",
    model.wpe.weight
)

# ------------------------------------------------------------------
# Final LN
# ------------------------------------------------------------------

compare_weight(
    "ln_f.weight",
    model.ln_f.weight
)

compare_weight(
    "ln_f.bias",
    model.ln_f.bias
)

# ------------------------------------------------------------------
# All transformer blocks
# ------------------------------------------------------------------

for i, block in enumerate(model.h):

    print("\n" + "#" * 80)
    print(f"BLOCK {i}")
    print("#" * 80)

    compare_weight(
        f"h.{i}.ln_1.weight",
        block.ln_1.weight
    )

    compare_weight(
        f"h.{i}.ln_1.bias",
        block.ln_1.bias
    )

    compare_weight(
        f"h.{i}.attn.c_attn.weight",
        block.attn.c_attn.weight
    )

    compare_weight(
        f"h.{i}.attn.c_attn.bias",
        block.attn.c_attn.bias
    )

    compare_weight(
        f"h.{i}.attn.c_proj.weight",
        block.attn.c_proj.weight
    )

    compare_weight(
        f"h.{i}.attn.c_proj.bias",
        block.attn.c_proj.bias
    )

    compare_weight(
        f"h.{i}.ln_2.weight",
        block.ln_2.weight
    )

    compare_weight(
        f"h.{i}.ln_2.bias",
        block.ln_2.bias
    )

    compare_weight(
        f"h.{i}.mlp.c_fc.weight",
        block.mlp.c_fc.weight
    )

    compare_weight(
        f"h.{i}.mlp.c_fc.bias",
        block.mlp.c_fc.bias
    )

    compare_weight(
        f"h.{i}.mlp.c_proj.weight",
        block.mlp.c_proj.weight
    )

    compare_weight(
        f"h.{i}.mlp.c_proj.bias",
        block.mlp.c_proj.bias
    )


HF : wte.weight
ZG : wte
HF shape : (50257, 768)
ZG shape : (50257, 768)

HF first 10:
-0.110103, -0.039267, 0.033108, 0.133826, -0.048476, -0.078918, -0.239774, -0.089474, 0.025255, -0.107397

ZG first 10:
-0.110103, -0.039267, 0.033108, 0.133826, -0.048476, -0.078918, -0.239774, -0.089474, 0.025255, -0.107397

MAX DIFF: 0.000000e+00

HF : wpe.weight
ZG : wpe
HF shape : (1024, 768)
ZG shape : (1024, 768)

HF first 10:
-0.018821, -0.197419, 0.004027, 0.011347, 0.063824, -0.105013, 0.036937, -0.168030, -0.049111, -0.056461

ZG first 10:
-0.018821, -0.197419, 0.004027, 0.011347, 0.063824, -0.105013, 0.036937, -0.168030, -0.049111, -0.056461

MAX DIFF: 0.000000e+00

HF : ln_f.weight
ZG : ln.weight
HF shape : (768,)
ZG shape : (768,)

HF first 10:
1.397080, 1.374953, 1.886957, 1.168837, 1.272385, 1.250812, 9.419820, 1.437056, 1.452746, 1.185577

ZG first 10:
1.397080, 1.374953, 1.886957, 1.168837, 1.272385, 1.250812, 9.419820, 1.437056, 1.452746, 1.185577

MAX DIFF: 0.000000e+00

HF : ln_

In [12]:
from transformers import GPT2Model

model = GPT2Model.from_pretrained("gpt2")

print(model.h[0].mlp.c_proj.weight.shape)
print(model.h[0].attn.c_attn.bias.shape)

torch.Size([3072, 768])
torch.Size([2304])


In [3]:
import torch
from transformers import GPT2Model
from safetensors.torch import load_file

c_path = "/Users/uonliaquat/workspace/zerograd/my_model.safetensors"
c = load_file(c_path)

print("=" * 60)
print("KEYS IN YOUR C FILE:")
for k in c:
    print(f"  {k:40s} shape={tuple(c[k].shape)}")
print("=" * 60)

# HF reference, same input: 3 positions, all token id 0
m = GPT2Model.from_pretrained("gpt2").eval()
with torch.no_grad():
    wte = m.wte(torch.zeros(3, dtype=torch.long))
    wpe = m.wpe(torch.arange(3))
    embed = wte + wpe

def row(t, p):
    t = t.detach().cpu().flatten() if t.dim() == 1 else t[p].detach().cpu()
    return ", ".join(f"{float(v):.3f}" for v in t.flatten()[:10])

print("\nHF REFERENCE (token id 0 at each position):")
print(f"  wpe[0]   : {row(m.wpe.weight, 0)}")
print(f"  wpe[1]   : {row(m.wpe.weight, 1)}")
print(f"  wpe[2]   : {row(m.wpe.weight, 2)}")
print(f"  embed[0] : {row(embed, 0)}   (= wte[0] + wpe[0])")
print(f"  embed[1] : {row(embed, 1)}   (= wte[0] + wpe[1])")
print(f"  embed[2] : {row(embed, 2)}   (= wte[0] + wpe[2])")

# Try to find your embedding tensor automatically
print("\nYOUR C TENSORS (first 10 at positions 0,1,2):")
candidates = [k for k in c if any(s in k.lower() for s in
              ("embed", "wpe", "wte", "input", "resid", "ln1"))]
if not candidates:
    candidates = list(c.keys())

for k in candidates:
    t = c[k]
    print(f"\n  {k}  shape={tuple(t.shape)}")
    if t.dim() >= 2 and t.shape[0] >= 3:
        print(f"    pos0: {row(t, 0)}")
        print(f"    pos1: {row(t, 1)}")
        print(f"    pos2: {row(t, 2)}")
    else:
        print(f"    flat: {row(t.flatten(), 0)}")

KEYS IN YOUR C FILE:
  token.ids                                shape=(1024,)
  wte                                      shape=(50257, 768)
  token.indices                            shape=(1024,)
  wpe                                      shape=(1024, 768)
  token.embed                              shape=(1024, 768)
  pos.embed                                shape=(1024, 768)
  input.embed                              shape=(1024, 768)
  h.0.ln1.weight                           shape=(768,)
  h.0.ln1.bias                             shape=(768,)
  h.0.ln1.out                              shape=(1024, 768)
  h.0.qkv.weight                           shape=(768, 2304)
  h.0.qkv.bias                             shape=(2304,)
  h.0.qkv.proj                             shape=(1024, 2304)
  h.0.attn.out                             shape=(1024, 768)
  h.0.attn.proj.weight                     shape=(768, 768)
  h.0.attn.proj.bias                       shape=(768,)
  h.0.attn.proj              

In [4]:
import torch, math
from transformers import GPT2Model
from safetensors.torch import load_file

c = load_file("/Users/uonliaquat/workspace/zerograd/my_model.safetensors")
m = GPT2Model.from_pretrained("gpt2").eval()

S = 1024
ids = torch.zeros((1, S), dtype=torch.long)

def cmp(name, hf, key):
    hf = hf.detach().reshape(-1)
    ct = c[key].reshape(-1)
    n = min(hf.numel(), ct.numel())
    d = (hf[:n] - ct[:n]).abs().max().item()
    flag = "  <-- MISMATCH" if d > 1e-3 else ""
    print(f"{name:28s} maxdiff={d:.5f}{flag}")

with torch.no_grad():
    x = m.wte(ids) + m.wpe(torch.arange(S).unsqueeze(0))
    cmp("input.embed", x, "input.embed")

    b = m.h[0]
    ln1 = b.ln_1(x)
    cmp("h.0.ln1.out", ln1, "h.0.ln1.out")

    qkv = b.attn.c_attn(ln1)
    cmp("h.0.qkv.proj", qkv, "h.0.qkv.proj")

    # full HF attention for block 0
    attn_out = b.attn(ln1)[0]
    cmp("h.0.attn.proj", attn_out, "h.0.attn.proj")

    resid1 = x + attn_out
    cmp("h.0.res.conn1", resid1, "h.0.res.conn1")

    ln2 = b.ln_2(resid1)
    cmp("h.0.ln2.out", ln2, "h.0.ln2.out")

    up = b.mlp.c_fc(ln2)
    cmp("h.0.mlp.up.proj", up, "h.0.mlp.up.proj")

    gelu = b.mlp.act(up)
    cmp("h.0.mlp.gelu", gelu, "h.0.mlp.up.proj.gelu.out")

    down = b.mlp.c_proj(gelu)
    cmp("h.0.mlp.down.proj", down, "h.0.mlp.down.proj")

    out = resid1 + down
    cmp("h.0.out", out, "h.0.out")

input.embed                  maxdiff=0.00000
h.0.ln1.out                  maxdiff=0.00035
h.0.qkv.proj                 maxdiff=0.00272  <-- MISMATCH
h.0.attn.proj                maxdiff=37.18219  <-- MISMATCH
h.0.res.conn1                maxdiff=37.18219  <-- MISMATCH
h.0.ln2.out                  maxdiff=2.41908  <-- MISMATCH
h.0.mlp.up.proj              maxdiff=15.06082  <-- MISMATCH
h.0.mlp.gelu                 maxdiff=11.50269  <-- MISMATCH
h.0.mlp.down.proj            maxdiff=105.23499  <-- MISMATCH
h.0.out                      maxdiff=105.10256  <-- MISMATCH
